# Suite PRT — Printing

The standards print any Ccpp node: a translation unit as a file, anything else as the text it stands for. Trees
built by hand print as source text without help: parentheses are added where precedence needs them, and around
declarators that would otherwise bind wrongly. Printing validates the tree and checks it against the standard first.

In [ ]:
import { Ccpp20, Syntax as S } from "@mbse/programs/Ccpp";
import { CcppStandard } from "@mbse/programs/Ccpp/_Standard";
import { Errors, type Syntax as F } from "@mbse/programs/Framework";

import { assert, equal, raises } from "./support.js";

const { PrintError } = Errors;
const [CPP26, C23] = [new CcppStandard(2026), new CcppStandard(2023, "C")];
const P = (node: F.Node) => CPP26.print(node);
const n = (spelling: string) => new S.Identifier({ spelling });
const e = (spelling: string) => new S.IdExpression({ name: n(spelling) });
const i = (value: number) => new S.IntegerLiteral({ spelling: String(value) });
const b = (left: F.Node, operator: string, right: F.Node) => new S.BinaryExpression({ left, operator, right });
const prim = (...keywords: string[]) => keywords.map((keyword) => new S.PrimitiveTypeSpecifier({ keyword }));
const t = (keyword: string, declarator: F.Node | null = null) => new S.TypeId({ specifiers: prim(keyword), declarator });
const d = (spelling: string) => new S.IdDeclarator({ name: n(spelling) });
const v = (declarator: F.Node, initializer: F.Node | null = null) => new S.SimpleDeclaration({ specifiers: prim("int"),
  declarators: [new S.InitDeclarator({ declarator, initializer })] });
const block = (...items: F.Node[]) => new S.CompoundStatement({ items });
const stmt = (expression: F.Node) => new S.ExpressionStatement({ expression });
const attrs = () => [new S.StandardAttributeSpecifier()];

## PRT-01 · Precedence: operands are parenthesized where they bind less than their place needs

In [ ]:
equal(P(b(b(e("a"), "+", e("b")), "*", e("c"))), "(a + b) * c");
equal(P(b(e("a"), "-", b(e("b"), "-", e("c")))), "a - (b - c)");
equal(P(b(b(e("a"), "-", e("b")), "-", e("c"))), "a - b - c");
const assign = () => new S.AssignmentExpression({ left: e("a"), operator: "=",
  right: new S.AssignmentExpression({ left: e("b"), operator: "=", right: e("c") }) });
equal(P(assign()), "a = b = c");
equal(P(new S.AssignmentExpression({ left: assign(), operator: "+=", right: i(1) })), "(a = b = c) += 1");
equal(P(new S.ConditionalExpression({ condition: assign(), consequence: b(e("x"), ",", e("y")), alternative: assign() })),
  "(a = b = c) ? x, y : a = b = c");
equal(P(new S.CallExpression({ function: b(e("f"), "+", e("g")), arguments: [b(e("a"), ",", e("b")), t("int")] })),
  "(f + g)((a, b), int)");
equal(P(new S.UnaryExpression({ operator: "-", operand: new S.UnaryExpression({ operator: "-", operand: e("x") }) })), "- -x");
equal(P(new S.UnaryExpression({ operator: "-", operand: i(-1) })), "- -1");
equal(P(new S.UnaryExpression({ operator: "!", operand: new S.UnaryExpression({ operator: "-", operand: e("x") }) })), "!-x");
equal(P(new S.UnaryExpression({ operator: "*", operand: new S.PostfixExpression({ operand: e("p"), operator: "++" }) })), "*p++");
equal(P(new S.PostfixExpression({ operand: new S.UnaryExpression({ operator: "*", operand: e("p") }), operator: "++" })), "(*p)++");
equal(P(new S.MemberExpression({ object: b(e("a"), "+", e("b")), operator: ".", member: n("c") })), "(a + b).c");
equal(P(new S.CastExpression({ type: t("int"), operand: b(e("a"), "*", e("b")) })), "(int)(a * b)");
equal(P(new S.SizeofExpression({ operand: b(e("a"), "+", e("b")) })), "sizeof(a + b)");
equal(P(new S.SizeofExpression({ operand: e("a") })), "sizeof a");
equal(P(new S.DeleteExpression({ operand: b(e("a"), "+", e("b")) })), "delete (a + b)");
equal(P(b(e("a"), ".*", b(e("b"), "+", e("c")))), "a.*(b + c)");
equal(P(new S.ThrowExpression({ operand: b(e("a"), ",", e("b")) })), "throw (a, b)");
equal(P(new S.FoldExpression({ left: b(e("a"), "+", e("b")), operator: "+" })), "((a + b) + ...)");
equal(P(new S.PackExpansion({ pattern: b(e("a"), "+", e("b")) })), "(a + b)...");
const template = new S.TemplateId({ name: n("X"), arguments: [b(i(1), ">", i(2)), b(i(1), "<", i(2)), b(e("a"), ",", e("b")),
  new S.ParenthesizedExpression({ expression: b(i(3), ">>", i(1)) })] });
equal(P(new S.IdExpression({ name: template })), "X<(1 > 2), 1 < 2, (a, b), (3 >> 1)>");
const cT = () => new S.IdExpression({ name: new S.TemplateId({ name: n("C"), arguments: [t("int")] }) });
equal(P(new S.TemplateDeclaration({ requires: b(b(cT(), "||", cT()), "&&", new S.CallExpression({ function: e("f") })),
  declaration: v(d("x")) })), "template <> requires (C<int> || C<int>) && (f())\nint x;");
console.log("ok");

## PRT-02 · Declarators: inside out, with parentheses where a pointer binds less than a function or array

In [ ]:
equal(P(v(new S.FunctionDeclarator({ declarator: new S.PointerDeclarator({ declarator: d("f") }),
  parameters: [new S.ParameterDeclaration({ specifiers: prim("int") })] }))), "int (*f)(int);");
equal(P(v(new S.PointerDeclarator({ declarator: new S.FunctionDeclarator({ declarator: d("f") }) }))), "int *f();");
equal(P(v(new S.ArrayDeclarator({ declarator: new S.PointerDeclarator({ declarator: d("a") }), size: i(3) }))), "int (*a)[3];");
equal(P(v(new S.ReferenceDeclarator({ declarator: new S.ArrayDeclarator({ declarator: d("r"), size: i(3) }) }))), "int &r[3];");
equal(P(v(new S.ArrayDeclarator({ declarator: new S.ReferenceDeclarator({ rvalue: true, declarator: d("r") }) }))), "int (&&r)[];");
const qualified = new S.PointerDeclarator({ qualifiers: [new S.CvQualifier({ keyword: "const" })],
  declarator: new S.PointerDeclarator({ attributes: [new S.StandardAttributeSpecifier({ attributes: [new S.Attribute({ name: "a" })] })],
    declarator: d("p") }) });
equal(P(v(qualified)), "int *const *[[a]] p;");
equal(P(v(new S.ReferenceDeclarator({ attributes: attrs(), declarator: d("r") }))), "int &[[]] r;");
equal(P(v(new S.PointerDeclarator({ qualifiers: [new S.CvQualifier({ keyword: "volatile" })] }))), "int *volatile;");
equal(P(v(new S.PointerDeclarator({ scope: n("C"), declarator: d("m") }))), "int C::*m;");
equal(P(t("int", new S.FunctionDeclarator({ declarator: new S.PointerDeclarator(),
  parameters: [new S.ParameterDeclaration({ specifiers: prim("char") })] }))), "int(*)(char)");
equal(P(t("int", new S.ReferenceDeclarator({ declarator: new S.ArrayDeclarator({ size: i(2) }) }))), "int&[2]");
equal(P(t("int", new S.ArrayDeclarator({ declarator: new S.ReferenceDeclarator(), size: i(2) }))), "int(&)[2]");
equal(P(t("char", new S.PackDeclarator())), "char...");
equal(P(new S.ParameterDeclaration({ specifiers: prim("int"), declarator: new S.PackDeclarator({ declarator: d("xs") }) })), "int... xs");
equal(P(new S.ParameterDeclaration({ this_keyword: true, specifiers: [new S.NamedTypeSpecifier({ name: n("Self") })],
  declarator: new S.ReferenceDeclarator({ rvalue: true, declarator: d("self") }) })), "this Self &&self");
equal(P(new S.SimpleDeclaration({ specifiers: [new S.PlaceholderTypeSpecifier()], declarators: [new S.InitDeclarator({
  declarator: new S.FunctionDeclarator({ declarator: d("f"), ref_qualifier: "&&", qualifiers: [new S.CvQualifier({ keyword: "const" })],
    exception: new S.NoexceptSpecifier({ condition: new S.BooleanLiteral({ value: true }) }), trailing_return: t("int") }) })] })),
"auto f() const && noexcept(true) -> int;");
equal(P(new S.TypeId({ specifiers: prim("int") })), "int");
console.log("ok");

## PRT-03 · Every kind prints on its own, as the text it stands for

In [ ]:
const cases: [F.Node, string][] = [
  [new S.OperatorName({ operator: "new[]" }), "operator new[]"], [new S.OperatorName({ operator: "co_await" }), "operator co_await"],
  [new S.OperatorName({ operator: "+" }), "operator+"], [new S.ConversionName({ type: t("int") }), "operator int"],
  [new S.LiteralOperatorName({ suffix: "_km" }), 'operator""_km'], [new S.DestructorName({ type: n("T") }), "~T"],
  [new S.DestructorName({ type: new S.DecltypeSpecifier({ expression: e("x") }) }), "~decltype(x)"],
  [new S.QualifiedName({ global_scope: true, qualifiers: [n("a"), new S.TemplateId({ name: n("b"), arguments: [i(1)] })],
    name: new S.TemplateId({ template_keyword: true, name: n("c") }) }), "::a::b<1>::template c<>"],
  [new S.CharacterLiteral({ prefix: "U", text: "x" }), "U'x'"], [new S.StringLiteral({ text: "s" }), '"s"'],
  [new S.RawStringLiteral({ text: "r" }), 'R"(r)"'], [new S.RawStringLiteral({ prefix: "L", delimiter: "d", text: "r" }), 'LR"d(r)d"'],
  [new S.UserDefinedLiteral({ literal: new S.FloatingLiteral({ spelling: "1.5" }), suffix: "_m" }), "1.5_m"],
  [new S.ConcatenatedString({ parts: [new S.StringLiteral({ text: "a" }), e("B")] }), '"a" B'],
  [new S.BooleanLiteral(), "false"], [new S.NullptrLiteral(), "nullptr"], [new S.ThisExpression(), "this"],
  [new S.ParenthesizedExpression({ expression: e("x") }), "(x)"],
  [new S.DefaultCapture({ mode: "&" }), "&"], [new S.SimpleCapture({ name: n("x"), pack: true }), "x..."],
  [new S.ThisCapture({ copy: true }), "*this"],
  [new S.InitCapture({ by_reference: true, pack: true, name: n("x"), initializer: new S.InitializerList({ items: [i(1)] }) }), "&...x{1}"],
  [new S.RequiresExpression(), "requires {}"],
  [new S.SimpleRequirement({ expression: e("x") }), "x;"], [new S.TypeRequirement({ name: n("T") }), "typename T;"],
  [new S.CompoundRequirement({ expression: e("x") }), "{ x };"],
  [new S.CompoundRequirement({ expression: e("x"), noexcept: true, return_type: n("C") }), "{ x } noexcept -> C;"],
  [new S.NestedRequirement({ constraint: e("C") }), "requires C;"],
  [new S.PackIndexingExpression({ pack: n("xs"), index: i(0) }), "xs...[0]"],
  [new S.ReflectExpression(), "^^::"], [new S.ReflectExpression({ operand: n("std") }), "^^std"],
  [new S.ReflectExpression({ operand: t("int") }), "^^int"], [new S.ReflectExpression({ operand: b(e("a"), "+", e("b")) }), "^^(a + b)"],
  [new S.SpliceExpression({ reflection: e("r") }), "[: r :]"],
  [new S.SpliceExpression({ template_keyword: true, reflection: e("r"), arguments: [t("int")] }), "template [: r :]<int>"],
  [new S.SubscriptExpression({ object: e("m"), indices: [i(1), i(2)] }), "m[1, 2]"],
  [new S.FunctionalCastExpression({ type: new S.PlaceholderTypeSpecifier(),
    initializer: new S.ParenthesizedInitializer({ arguments: [e("x")] }) }), "auto(x)"],
  [new S.TypeidExpression({ operand: t("int") }), "typeid(int)"], [new S.AwaitExpression({ operand: e("x") }), "co_await x"],
  [new S.SizeofPackExpression({ pack: n("Ts") }), "sizeof...(Ts)"],
  [new S.AlignofExpression({ keyword: "__alignof__", operand: e("x") }), "__alignof__(x)"],
  [new S.NoexceptExpression({ operand: e("x") }), "noexcept(x)"],
  [new S.NewExpression({ parenthesized_type: true, type: t("int"), global_scope: true, placement: [e("p")] }), "::new (p) (int)"],
  [new S.DeleteExpression({ array: true, global_scope: true, operand: e("p") }), "::delete[] p"],
  [new S.ThrowExpression(), "throw"], [new S.YieldExpression({ operand: e("x") }), "co_yield x"],
  [new S.PackExpansion({ pattern: t("int") }), "int..."],
  [new S.InitializerList({ trailing_comma: true, items: [i(1)] }), "{1,}"],
  [new S.DesignatedInitializer({ designators: [new S.FieldDesignator({ name: n("x") }), new S.IndexDesignator({ index: i(1), last: i(2) })],
    initializer: new S.InitializerList() }), ".x[1 ... 2]{}"],
  [new S.FieldDesignator({ name: n("y") }), ".y"], [new S.IndexDesignator({ index: i(0) }), "[0]"],
  [new S.StatementExpression({ body: block() }), "({})"], [new S.ExtensionExpression({ operand: e("x") }), "__extension__ x"],
  [new S.DefinedExpression({ name: n("X") }), "defined(X)"],
  [new S.DeclSpecifier({ keyword: "static" }), "static"], [new S.ExplicitSpecifier(), "explicit"],
  [new S.CvQualifier({ keyword: "const" }), "const"], [new S.TypenameSpecifier({ name: n("T") }), "typename T"],
  [new S.PlaceholderTypeSpecifier({ constraint: n("C"), decltype: true }), "C decltype(auto)"],
  [new S.TypeofSpecifier({ keyword: "__typeof__", operand: t("int") }), "__typeof__(int)"],
  [new S.PackIndexingSpecifier({ pack: n("Ts"), index: i(1) }), "Ts...[1]"],
  [new S.SpliceSpecifier({ typename_keyword: true, reflection: e("r") }), "typename [: r :]"],
  [new S.ClassSpecifier({ key: "struct" }), "struct"], [new S.EnumSpecifier({ key: "enum", body: new S.EnumeratorList() }), "enum {}"],
  [new S.BaseSpecifier({ attributes: attrs(), type: n("B") }), "[[]] B"],
  [new S.IdDeclarator({ name: n("x") }), "x"],
  [new S.StructuredBindingDeclarator({ bindings: [d("a"), new S.PackDeclarator({ declarator: d("r") })] }), "[a, ...r]"],
  [new S.ParenthesizedDeclarator({ declarator: d("x") }), "(x)"],
  [new S.EllipsisParameter(), "..."],
  [new S.TypeParameter(), "typename"], [new S.TypeParameter({ constraint: n("C"), pack: true, name: n("Ts") }), "C... Ts"],
  [new S.TypeParameter({ key: "class", name: n("T"), default: t("int") }), "class T = int"],
  [new S.TemplateTemplateParameter({ key: "concept", requires: e("B"), name: n("C"), default: n("D"),
    parameters: [new S.TypeParameter()] }), "template <typename> requires B concept C = D"],
  [new S.EqualInitializer({ value: i(1) }), "= 1"], [new S.ParenthesizedInitializer({ arguments: [i(1), i(2)] }), "(1, 2)"],
  [new S.NoexceptSpecifier(), "noexcept"], [new S.ThrowSpecifier(), "throw()"],
  [new S.PreconditionSpecifier({ predicate: e("x") }), "pre(x)"],
  [new S.PostconditionSpecifier({ attributes: attrs(), result: n("r"), predicate: e("r") }), "post [[]](r: r)"],
  [new S.PostconditionSpecifier({ predicate: e("x") }), "post(x)"],
  [new S.StandardAttributeSpecifier({ using_namespace: "gnu", attributes: [new S.Attribute({ name: "a", pack: true }),
    new S.Annotation({ value: i(1), pack: true })] }), "[[using gnu: a..., =1...]]"],
  [new S.Attribute({ namespace: "gnu", name: "cold", arguments: [i(1), t("int")] }), "gnu::cold(1, int)"],
  [new S.Annotation({ value: i(2) }), "=2"], [new S.AlignasSpecifier({ keyword: "alignas", operand: e("Ts"), pack: true }), "alignas(Ts...)"],
  [new S.GnuAttributeSpecifier({ keyword: "__attribute", attributes: [new S.Attribute({ name: "pure" })] }), "__attribute((pure))"],
  [new S.DeclspecSpecifier({ attributes: [new S.Attribute({ name: "a" }), new S.Attribute({ name: "b" })] }), "__declspec(a b)"],
  [new S.InitDeclarator({ declarator: d("x"), bitfield: i(3), initializer: new S.InitializerList() }), "x : 3{}"],
  [new S.MemberInitializer({ member: n("Bs"), initializer: new S.ParenthesizedInitializer({ arguments: [e("x")] }), pack: true }), "Bs(x)..."],
  [new S.VirtSpecifier({ keyword: "final" }), "final"], [new S.DefaultedBody(), "= default;"], [new S.DeletedBody(), "= delete;"],
  [new S.DeletedBody({ reason: new S.StringLiteral({ text: "why" }) }), '= delete("why");'],
  [new S.MemberList(), "{}"], [new S.MemberList({ items: [v(d("x"))] }), "{\n    int x;\n}"],
  [new S.EnumeratorList({ enumerators: [new S.Enumerator({ name: n("A") })] }), "{\n    A\n}"],
  [new S.Enumerator({ name: n("A"), attributes: attrs(), value: i(1) }), "A [[]] = 1"],
  [new S.NamespaceName({ inline: true, name: n("v1") }), "inline v1"], [new S.NamespaceName({ name: n("v1") }), "v1"],
  [new S.UsingDeclarator({ typename_keyword: true, name: n("T"), pack: true }), "typename T..."],
  [new S.Handler({ parameter: new S.EllipsisParameter(), body: block() }), "catch (...) {}"],
  [new S.AsmOperand({ constraint: new S.StringLiteral({ text: "r" }), value: e("x") }), '"r" (x)'],
  [new S.GenericAssociation({ value: i(1) }), "default: 1"], [new S.GenericAssociation({ type: t("int"), value: i(1) }), "int: 1"],
  [new S.Comment({ text: " c" }), "// c"], [new S.Comment({ block: true, text: " c " }), "/* c */"],
  [new S.LambdaDeclarator(), "()"],
  [new S.LambdaDeclarator({ contracts: [new S.PreconditionSpecifier({ predicate: e("x") })], requires: e("C"), attributes: attrs() }),
    "() [[]] requires C pre(x)"],
];
for (const [node, expected] of cases) {
  const standard = [CPP26, C23, new CcppStandard(2017)].find((s) => s.check(node).length === 0) as CcppStandard; // C in C
  const printed = standard.print(node);
  assert(printed === expected, `${node.kind().KIND}: ${printed} != ${expected}`);
}
console.log(new Set(cases.map(([node]) => node.kind())).size, "kinds on their own");

## PRT-04 · Statements and declarations: layout, and the constructs no parser writes yet

In [ ]:
equal(P(new S.IfStatement({ consteval: true, negated: true, consequence: block(), alternative: block() })), "if !consteval {} else {}");
equal(P(new S.IfStatement({ consteval: true, consequence: block() })), "if consteval {}");
equal(P(new S.IfStatement({ condition: e("a"), consequence: new S.BreakStatement(), alternative: new S.IfStatement({
  condition: e("b"), consequence: new S.BreakStatement() }) })), "if (a)\n    break;\nelse if (b)\n    break;");
equal(P(new S.DoStatement({ body: new S.ExpressionStatement(), condition: e("x") })), "do\n    ;\nwhile (x);");
equal(P(new S.ForStatement({ body: block() })), "for (;;) {}");
equal(P(new S.ForStatement({ condition: v(d("x"), new S.EqualInitializer({ value: i(1) })), body: block() })), "for (; int x = 1;) {}");
equal(P(new S.RangeForStatement({ template_keyword: true, declaration: new S.SimpleDeclaration({
  specifiers: [new S.PlaceholderTypeSpecifier()], declarators: [new S.InitDeclarator({ declarator: d("x") })] }),
range: new S.InitializerList({ items: [i(1)] }), body: block() })), "template for (auto x : {1}) {}");
equal(P(new S.ContractAssertStatement({ attributes: attrs(), predicate: e("x") })), "contract_assert [[]](x);");
equal(P(new S.AttributedStatement({ attributes: [new S.StandardAttributeSpecifier({ attributes: [new S.Attribute({ name: "likely" })] })],
  statement: block() })), "[[likely]] {}");
equal(P(new S.AttributedStatement({ attributes: attrs(), statement: new S.LabeledStatement({ label: n("l") }) })), "[[]] l:");
const sw = new S.SwitchStatement({ condition: e("x"), body: block(
  new S.CaseStatement({ value: i(1), last: i(3), statement: stmt(e("a")) }), stmt(e("b")),
  new S.CaseStatement({ value: i(4) }), new S.DefaultStatement({ statement: new S.DefaultStatement() })) });
equal(P(sw), "switch (x) {\n    case 1 ... 3:\n        a;\n        b;\n    case 4:\n    default:\n    default:\n}");
const labelled = () => new S.LabeledStatement({ label: n("l"), statement: new S.DefaultStatement({ statement: v(d("x")) }) });
equal(P(labelled()), "l:\ndefault:\nint x;");
equal(P(block(labelled())), "{\nl:\ndefault:\n    int x;\n}"); // labels are outdented
equal(P(new S.IfStatement({ condition: e("x"), consequence: new S.LabeledStatement({ label: n("l"), statement: block() }) })),
  "if (x)\nl: {}");
equal(P(new S.FunctionDefinition({ specifiers: prim("void"), declarator: new S.FunctionDeclarator({ declarator: d("f") }),
  requires: cT(), contracts: [new S.PreconditionSpecifier({ predicate: e("x") }), new S.PostconditionSpecifier({ predicate: e("y") })],
  body: new S.DeletedBody({ reason: new S.StringLiteral({ text: "no" }) }) })), 'void f() requires C<int> pre(x) post(y) = delete("no");');
equal(P(new S.SimpleDeclaration({ specifiers: prim("void"), declarators: [new S.InitDeclarator({
  declarator: new S.FunctionDeclarator({ declarator: d("g") }), requires: cT(),
  contracts: [new S.PreconditionSpecifier({ predicate: e("x") })] })] })), "void g() requires C<int> pre(x);");
equal(P(new S.FunctionDefinition({ declarator: new S.FunctionDeclarator({ declarator: d("S") }), initializers: [
  new S.MemberInitializer({ member: n("x"), initializer: new S.InitializerList() })], body: new S.DefaultedBody() })),
"S() : x{} = default;");
equal(P(new S.FriendTypeDeclaration({ types: [t("int"), new S.PackExpansion({ pattern: new S.TypeId({
  specifiers: [new S.NamedTypeSpecifier({ name: n("Ts") })] }) })] })), "friend int, Ts...;");
equal(P(new S.UsingDeclaration({ declarators: [new S.UsingDeclarator({ name: n("a") }), new S.UsingDeclarator({ name: n("b") })] })),
  "using a, b;");
equal(P(new S.ConceptDefinition({ name: n("C"), attributes: attrs(), constraint: i(1) })), "concept C [[]] = 1;");
equal(P(new S.AliasDeclaration({ name: n("T"), attributes: attrs(), type: t("int") })), "using T [[]] = int;");
equal(P(new S.ImportDeclaration({ name: "m", attributes: attrs() })), "import m [[]];");
equal(P(new S.ModuleDeclaration({ name: "m" })), "module m;");
equal(P(new S.UsingDirective({ attributes: attrs(), name: n("std") })), "[[]] using namespace std;");
equal(P(new S.LinkageSpecification({ language: "C", items: [v(d("x"))] })), 'extern "C" int x;');
equal(P(new S.AsmDeclaration({ attributes: attrs(), keyword: "asm", template: new S.StringLiteral({ text: "" }),
  inputs: [new S.AsmOperand({ name: n("x"), constraint: new S.StringLiteral({ text: "r" }), value: e("x") })] })),
'[[]] asm("" : : [x] "r" (x));');
equal(P(new S.ClassSpecifier({ key: "struct", attributes: attrs(), name: n("S"),
  body: new S.MemberList({ items: [new S.AccessSpecifier({ access: "public" })] }) })), "struct [[]] S {\npublic:\n}");
equal(P(new S.EnumSpecifier({ key: "enum class", attributes: attrs(), name: n("E"), base: t("int") })), "enum class [[]] E : int");
equal(P(new S.SimpleDeclaration({ specifiers: prim("int"), declarators: [new S.InitDeclarator({ bitfield: i(3) })] })), "int : 3;");
equal(P(new S.TranslationUnit()), "");
equal(P(new S.TranslationUnit({ items: [new S.EmptyDeclaration()] })), ";\n");
console.log("ok");

## PRT-05 · Directives, comments and enumerators

In [ ]:
const enumerators = new S.EnumeratorList({ enumerators: [
  new S.Comment({ text: " first" }), new S.Enumerator({ name: n("A") }), new S.Comment({ text: " after A", trailing: true }),
  new S.IfdefDirective({ name: n("X"), items: [new S.Enumerator({ name: n("B") })], alternative: new S.ElseDirective({
    items: [new S.Enumerator({ name: n("C") }), new S.Comment({ text: " c", trailing: true })] }) }),
  new S.Enumerator({ name: n("D") }), new S.Comment({ text: " last", block: true })] });
equal(P(enumerators), "{\n    // first\n    A, // after A\n#ifdef X\n    B,\n#else\n    C, // c\n#endif\n    D\n    /* last*/\n}");
equal(P(new S.EnumeratorList({ enumerators: [new S.Comment({ text: " none" })] })), "{\n    // none\n}");
equal(P(new S.EnumeratorList({ enumerators: [new S.Enumerator({ name: n("A") })], trailing_comma: true })), "{\n    A,\n}");
equal(P(new S.TranslationUnit({ items: [
  new S.IfDirective({ condition: new S.DefinedExpression({ name: n("X") }), items: [v(d("x"))], alternative: new S.ElifdefDirective({
    negated: true, name: n("Y"), items: [], alternative: new S.ElifDirective({ condition: i(1), items: [] }) }) }),
  new S.Comment({ text: " end", trailing: true }), new S.DefineDirective({ name: n("F"), function_like: true, variadic: true }),
  new S.OtherDirective({ directive: "pragma", text: "once" }), new S.OtherDirective(),
  new S.IncludeDirective({ directive: "include_next", path: "a.h" })] })),
"#if defined(X)\nint x;\n#elifndef Y\n#elif 1\n#endif // end\n#define F(...)\n#pragma once\n#\n#include_next \"a.h\"\n");
equal(P(new S.DefineDirective({ name: n("G"), function_like: true })), "#define G()");
equal(P(new S.IfdefDirective({ name: n("X"), items: [new S.Enumerator({ name: n("B") })] })), "#ifdef X\nB\n#endif"); // not in a list
console.log("ok");

## PRT-06 · Printing validates the tree and checks it against the standard

In [ ]:
raises(PrintError, () => P(new S.BinaryExpression({ left: e("a"), operator: "+" })), "a BinaryExpression needs a right");
raises(PrintError, () => P(new S.TranslationUnit({ items: [new S.Comment({ text: 1n })] })),
  "items[0]: Comment.text must be a str, got int");
raises(PrintError, () => new CcppStandard(2017).print(b(e("a"), "<=>", e("b"))),
  "BinaryExpression.operator '<=>' needs C++20, but this is C++17");
raises(PrintError, () => Ccpp20.print(b(e("a"), "+", new S.GenericSelection({ controlling: i(1) }))),
  "right: GenericSelection is not C++, but this is C++20");
equal(C23.print(new S.GenericSelection({ controlling: t("int"), associations: [new S.GenericAssociation({ value: i(1) })] })),
  "_Generic(int, default: 1)");
equal(C23.print(new S.TypeofSpecifier({ keyword: "typeof_unqual", operand: e("x") })), "typeof_unqual(x)");
equal(C23.print(new S.AtomicTypeSpecifier({ type: t("int") })), "_Atomic(int)");
equal(C23.print(new S.BitIntSpecifier({ width: i(7) })), "_BitInt(7)");
equal(C23.print(new S.CompoundLiteralExpression({ type: t("int"), initializer: new S.InitializerList({ items: [i(1)] }) })), "(int){1}");
console.log("ok");